# D-02. 미등록 판별이 안 되는 원인 확인 — 특성 MLP 임베딩

D-01 결과: 5개 방법 모두 평균 AUROC 0.44~0.55, carpet(far)이 가장 나쁨(proto 0.34)

가설: 임베딩이 재질보다 **녹화**를 더 많이 담고 있어서, 처음 보는 녹화의 알려진 재질도 빠진 재질만큼 프로토타입에서 멀다

확인 방법 (147 split마다 프로토타입 거리를 세 그룹으로 나눔)

| 그룹 | 뜻 |
|---|---|
| train | 프로토타입을 만든 학습 녹화 구간 (모델이 본 녹화) |
| known | 알려진 재질의 테스트 녹화 구간 (처음 보는 녹화) |
| unknown | 빠진 재질 구간 (처음 보는 재질) |

- `auroc_train_unknown`이 높다 → 임베딩이 본 것과 새 재질은 구분한다
- `auroc_train_known`도 그만큼 높다 → 처음 보는 녹화도 똑같이 낯설다 = **녹화 차이가 원인**
- `known_acc`: 처음 보는 녹화의 알려진 재질을 가장 가까운 프로토타입으로 맞힌 비율

실행: D-01과 같은 준비(임베딩 zip을 MyDrive/에), CPU로 충분

## 0. 환경 준비

In [ ]:
import os
import sys
from pathlib import Path

BRANCH = "D/ood-mlp"   # main에 merge된 뒤에는 "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/ML-imu").exists():
        !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
    os.chdir("/content/ML-imu")
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
    !pip install -q -r requirements.txt
    !python data/download.py

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path += [str(ROOT / "src"), str(ROOT / "notebooks" / "D_ood")]
print(ROOT)

In [ ]:
# 임베딩 위치: 받은 zip을 내 드라이브 맨 위(MyDrive/)에 올려 두면 Colab 안에 풀어서 쓴다
# (드라이브에 풀린 폴더 MyDrive/ML-imu_embeddings가 있으면 그 폴더를 바로 쓴다)
EMB_ZIP = "ML-imu_embeddings*.zip"

if IN_COLAB:
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    OUT = MYDRIVE / "ML-imu_D"
    EMB = MYDRIVE / "ML-imu_embeddings"
    if not EMB.exists():
        EMB = ROOT / "embeddings" / "mlp"
        if not any(EMB.glob("emb_mlp_*.npy")):
            zips = sorted(MYDRIVE.glob(EMB_ZIP))
            assert zips, f"MyDrive/에 {EMB_ZIP} 파일을 올려 주세요"
            EMB.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zips[-1]) as zf:
                for info in zf.infolist():
                    if not info.is_dir():
                        (EMB / Path(info.filename).name).write_bytes(zf.read(info))
            print("압축 해제:", zips[-1].name)
else:
    EMB = ROOT / "embeddings" / "mlp"
    OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

n_emb = len(list(EMB.glob("emb_mlp_*.npy")))
print(EMB, n_emb, "개 임베딩 (기대값 168 = 147 + 21)")

## 1. 데이터

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recordings import load_dataset, CHANNEL_NAMES
from splits import load_material_folds
from ood import Prototype, diagnose_fold, DIFFICULTY

X, y, rec = load_dataset("data")
folds = load_material_folds()
vib = X[:, :, CHANNEL_NAMES.index("acc_z")].std(axis=1)   # 구간별 진동 세기 (가속도 Z 표준편차)

MODEL = "mlp"
def load_emb(held_out, fold):
    return np.load(EMB / f"emb_{MODEL}_{held_out}_{fold}.npy")

order = sorted({f["held_out"] for f in folds}, key=lambda m: ({"far": 0, "mid": 1, "near": 2}[DIFFICULTY.get(m, "mid")], m))
print(X.shape, len(folds), "개 split")

## 2. 세 그룹 거리 비교 (147 split)

In [ ]:
rows = []
for split in folds:
    r = diagnose_fold(load_emb(split["held_out"], split["fold"]), y, rec, split)
    rows.append({"heldout": split["held_out"], "fold": split["fold"],
                 "known_rec": split["test_known"][0], "known_mat": y[rec == split["test_known"][0]][0], **r})
diag = pd.DataFrame(rows)
diag.to_csv(OUT / f"D_diagnosis_{MODEL}.csv", index=False)

cols = ["med_train", "med_known", "med_unknown", "auroc_train_unknown", "auroc_train_known", "auroc_known_unknown", "known_acc"]
tab = diag.groupby("heldout")[cols].mean().loc[order]
tab.loc["평균"] = tab.mean()
display(tab.round(3))

읽는 법
- `med_known`이 `med_train`보다 훨씬 크고 `med_unknown`과 비슷하면 → 처음 보는 녹화 자체가 낯설다
- `auroc_train_unknown`과 `auroc_train_known`이 둘 다 높고 비슷하면 → 녹화 차이가 미등록 신호를 덮는다

## 3. 그림: 세 그룹의 거리 분포 (빠진 재질별, fold 전체)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, m in zip(axes.flat, order):
    d = {"train": [], "known": [], "unknown": []}
    for split in [f for f in folds if f["held_out"] == m]:
        z = load_emb(m, split["fold"])
        train = np.isin(rec, split["train"])
        proto = Prototype().fit(z[train], y[train])
        idx = np.where(train)[0]
        idx = np.random.default_rng(42).choice(idx, size=min(500, len(idx)), replace=False)   # 학습 구간은 일부만
        d["train"].append(proto.score(z[idx]))
        d["known"].append(proto.score(z[np.isin(rec, split["test_known"])]))
        d["unknown"].append(proto.score(z[np.isin(rec, split["unknown"])]))
    d = {g: np.concatenate(v) for g, v in d.items()}
    hi = np.percentile(np.concatenate(list(d.values())), 99)   # 극단값 몇 개가 x축을 늘리지 않게 99%에서 자름
    for g, c in [("train", "tab:gray"), ("known", "tab:blue"), ("unknown", "tab:red")]:
        ax.hist(np.clip(d[g], 0, hi), bins=60, range=(0, hi), density=True, alpha=0.5, color=c, label=g)
    ax.set_title(f"held out: {m} ({DIFFICULTY.get(m, 'mid')})")
axes[0, 0].legend()
for ax in axes[1]:
    ax.set_xlabel("distance to nearest prototype")
plt.tight_layout()
plt.savefig(OUT / f"D_distance_hist_{MODEL}.png", dpi=150)
plt.show()

## 4. 처음 보는 녹화별: 거리와 분류 정확도

녹화마다 테스트로 쓰인 split(재질 7개를 뺀 경우)의 평균

In [ ]:
per_rec = (diag.groupby(["known_rec", "known_mat"])[["med_known", "known_acc", "auroc_known_unknown"]]
           .mean().reset_index().sort_values("med_known", ascending=False))
per_rec["rec_vib"] = per_rec["known_rec"].map(lambda r: float(np.median(vib[rec == r])))
per_rec.to_csv(OUT / f"D_per_recording_{MODEL}.csv", index=False)
display(per_rec.round(3))
print("녹화 거리와 정확도 상관 (Spearman):", round(per_rec[["med_known", "known_acc"]].corr("spearman").iloc[0, 1], 3))

## 5. 진동 세기와 거리

carpet이 거꾸로 나온 이유 확인: 조용한 구간은 프로토타입에 가깝고 덜컹거리는 구간은 먼가?

In [ ]:
from scipy.stats import spearmanr

rows = []
for m in order:
    split = next(f for f in folds if f["held_out"] == m and f["fold"] == 0)
    z = load_emb(m, 0)
    train = np.isin(rec, split["train"])
    proto = Prototype().fit(z[train], y[train])
    test = np.isin(rec, split["test_known"]) | np.isin(rec, split["unknown"])
    rho, _ = spearmanr(vib[test], proto.score(z[test]))
    rows.append({"heldout": m, "spearman_vib_dist": rho,
                 "vib_known": float(np.median(vib[np.isin(rec, split["test_known"])])),
                 "vib_unknown": float(np.median(vib[np.isin(rec, split["unknown"])]))})
vib_tab = pd.DataFrame(rows).set_index("heldout")
vib_tab.to_csv(OUT / f"D_vib_vs_dist_{MODEL}.csv")
display(vib_tab.round(3))

m = "carpet"
split = next(f for f in folds if f["held_out"] == m and f["fold"] == 0)
z = load_emb(m, 0)
train = np.isin(rec, split["train"])
proto = Prototype().fit(z[train], y[train])
fig, ax = plt.subplots(figsize=(7, 4))
for g, mask, c in [("known (unseen recording)", np.isin(rec, split["test_known"]), "tab:blue"),
                   ("unknown (carpet)", np.isin(rec, split["unknown"]), "tab:red")]:
    ax.scatter(vib[mask], proto.score(z[mask]), s=6, alpha=0.5, color=c, label=g)
ax.set_xlabel("vibration (acc_z std)"); ax.set_ylabel("distance to nearest prototype")
ax.set_title("held out: carpet, fold 0"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / f"D_vib_vs_dist_carpet_{MODEL}.png", dpi=150); plt.show()

## 메모할 것

- 녹화 차이 가설: `auroc_train_known` 평균 = ___, `auroc_train_unknown` 평균 = ___
- 처음 보는 녹화의 분류 정확도 평균 = ___, 가장 낯선 녹화 = ___
- 진동과 거리의 관계 / carpet이 거꾸로 나온 이유:
- 드라이브 산출물: `ML-imu_D/D_diagnosis_mlp.csv`, `D_distance_hist_mlp.png`, `D_vib_vs_dist_carpet_mlp.png`